In [1]:
!pip install -q bitsandbytes accelerate
!pip install -U bitsandbytes
!pip install -q git+https://github.com/huggingface/transformers.git@main
!pip install -q git+https://github.com/huggingface/peft.git@main

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import pandas as pd
from google.colab import files
import os
filename = "Q2_20230202_majority 1.csv"

if not os.path.exists(filename):
  print(f"'{filename}' not found. Please upload the file.")
  uploaded = files.upload()
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("Q2_20230202_majority 1.csv")
df.sample(5)

,tweet_id,created_at,tweet,label_majority,month
3650,1.384830e+18,2021-04-21 11:23:29+00:00,another pfizer blood clot victim in brisbane..,against,21-Apr
859,1.437975e+18,2021-09-15 03:02:08+00:00,"heartbreaking—5 kids are now orphaned after both mom+dad (age 37/39) dies of #covid19 weeks apart. the baby was born by c-section while mom was on a ventilator, doesn’t even have a name yet.😢 grandparents haven’t told the kids parents died yet. #vaccinate",in-favor,21-Sep
891,1.438913e+18,2021-09-17 17:09:27+00:00,"as an employer, i took the position that instituting mandatory vaccinations, changing conditions of employment, was against the law. it is a righteous position. it’s the right thing to do to not use that coercive tactic in my employees. i also can not break the law by asking my",against,21-Sep
1297,1.448793e+18,2021-10-14 23:30:05+00:00,i filed suit against saisd’s unlawful vaccine mandate and won! we are just getting started. #vaccinemandates,against,21-Oct
1918,1.467251e+18,2021-12-04 21:56:22+00:00,i was pretty tired today during training. i normally take the weekends off but with vacation coming up i figured i’d push through. i’m sluggish and got my booster shot and it’s starting to settle in 😪,in-favor,21-Dec


In [3]:
def format_prompt(tweet):

    examples = (
        f'Here are a few examples:'
        f'Tweet: "Vaccines saved my life" Stance: "in-favor"\n'
        f'Tweet: "Still deciding on those shots..." Stance: "neutral-or-unclear"\n'
        f'Tweet: "This vaccine is poison." Stance: "Against"\n'
        f'Now analyze:\n'
    )
    prompt = (
        f'What is the stance of the **author** of the following tweet toward COVID-19 vaccines?\n'
        f'Classify the stance as exactly one of: in-favor, against, neutral-or-unclear. If the stance is unclear or mixed or if the tweet is off-topic, choose "neutral-or-unclear".\n'
        f'{examples}'
        f'Tweet: "{tweet}"\n'
        f'Respond with one exactly of the following options: "in-favor", "against", "neutral-or-unclear".'
    )

    return prompt


df['prompt'] = df['tweet'].apply(format_prompt)
df['target'] = df['label_majority'].str.strip()
print("prompt loaded")

prompt loaded


In [4]:
df_phase1 = df[df['target'].isin(['in-favor', 'against'])]

df_phase2 = df[df['target'] == 'neutral-or-unclear']

df_phase3 = pd.concat([
    df[df['target'] == 'in-favor'].sample(n=1000, random_state=1),
    df[df['target'] == 'against'].sample(n=1000, random_state=1),
    df[df['target'] == 'neutral-or-unclear'].sample(n=1000, random_state=1),
])

print("Phases are prepared.")

Phases are prepared.


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")

def tokenize(example):
    inputs = tokenizer(example['prompt'], max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(example['target'], max_length=8, truncation=True, padding="max_length")
    inputs['labels'] = labels.input_ids
    return inputs

print("Tokenizer loaded.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer loaded.


In [6]:
from transformers import AutoModelForSeq2SeqLM
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model


model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large", load_in_8bit=True)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type="SEQ_2_SEQ_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"]
)
model = get_peft_model(model, lora_config)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [7]:
import shutil
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
from datasets import Dataset


def train_phase(df_phase, phase_name, num_epochs, lr):
    print(f"\n📘 Starting Phase: {phase_name} for {num_epochs} epoch(s) — LR: {lr}")

    dataset = Dataset.from_pandas(df_phase[['prompt', 'target']])
    tokenized = dataset.map(tokenize, batched=True)
    split = tokenized.train_test_split(test_size=0.2)
    train_dataset = split['train']
    eval_dataset = split['test']

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./results_{phase_name}",
        learning_rate=lr,

        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=num_epochs,
        # logging_steps=1,
        # eval_steps=250,
        # save_steps=250,
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir=f'./logs_{phase_name}',
        fp16=False,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        report_to="none"
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    print(f"✅ Phase {phase_name} training complete!")

    # Save & download
    model_path = f"./finetuned-flan-t5-vaccine_{phase_name}"
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    shutil.make_archive(model_path, 'zip', model_path)
    files.download(f"{model_path}.zip")

print("Training function ")

Training function 


In [9]:
train_phase(df_phase1, "phase1_easy", 2, lr=2e-5)


📘 Starting Phase: phase1_easy for 1 epoch(s) — LR: 2e-05


Map:   0%|          | 0/4711 [00:00<?, ? examples/s]

/tmp/ipython-input-7-4014438074.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autogra

Epoch,Training Loss,Validation Loss
1,15.487200,11.660830


✅ Phase phase1_easy training complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
train_phase(df_phase2, "phase2_neutral", 2, lr=1e-5)


📘 Starting Phase: phase2_neutral for 2 epoch(s) — LR: 1e-05


Map:   0%|          | 0/1040 [00:00<?, ? examples/s]

/tmp/ipython-input-7-4014438074.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autogra

Epoch,Training Loss,Validation Loss
1,0.178000,0.108069
2,0.094700,0.070240


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


✅ Phase phase2_neutral training complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# import zipfile
# import os

# model_dir = "finetuned-flan-t5-vaccine_phase2_neutral"
# zip_path = "/content/finetuned-flan-t5-vaccine_phase2_neutral.zip"

# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(model_dir)

In [11]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# from peft import PeftModel, PeftConfig

# # Load tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_dir)

# # Load base model config from the PEFT adapter
# peft_config = PeftConfig.from_pretrained(model_dir)
# base_model = AutoModelForSeq2SeqLM.from_pretrained(peft_config.base_model_name_or_path, load_in_8bit=True)

# # Prepare model for training with bitsandbytes
# from peft import prepare_model_for_kbit_training
# base_model = prepare_model_for_kbit_training(base_model)

# # Load PEFT (LoRA) adapter on top of the base model
# model = PeftModel.from_pretrained(base_model, model_dir)


In [12]:
train_phase(df_phase3, "phase3_mixed", 5, lr=2e-5)


📘 Starting Phase: phase3_mixed for 5 epoch(s) — LR: 2e-05


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

/tmp/ipython-input-7-4014438074.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autogra

Epoch,Training Loss,Validation Loss
1,5.636900,2.086684
2,1.849600,1.439169
3,1.483200,0.792759
4,1.132100,0.309041
5,0.957300,0.264586


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reen

✅ Phase phase3_mixed training complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
train_phase(df_phase3, "phase3_mixed", 1, lr=2e-5)


📘 Starting Phase: phase3_mixed for 1 epoch(s) — LR: 2e-05


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

/tmp/ipython-input-7-4014438074.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autogra

Epoch,Training Loss,Validation Loss
1,0.688400,0.168516


✅ Phase phase3_mixed training complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
train_phase(df_phase3, "phase3_mixed", 3, lr=2e-5)


📘 Starting Phase: phase3_mixed for 3 epoch(s) — LR: 2e-05


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

/tmp/ipython-input-7-4014438074.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autogra

Epoch,Training Loss,Validation Loss
1,0.434900,0.121199
2,0.315500,0.118033
3,0.296200,0.118088


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reen

✅ Phase phase3_mixed training complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
train_phase(df_phase3, "phase3_mixed", 3, lr=2e-5)


📘 Starting Phase: phase3_mixed for 3 epoch(s) — LR: 2e-05


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

/tmp/ipython-input-7-4014438074.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autogra

Epoch,Training Loss,Validation Loss
1,0.241200,0.111893
2,0.212200,0.112683
3,0.215900,0.113080


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reen

✅ Phase phase3_mixed training complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
train_phase(df_phase3, "phase3_mixed", 1, lr=1e-6)


📘 Starting Phase: phase3_mixed for 1 epoch(s) — LR: 1e-06


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

/tmp/ipython-input-7-4014438074.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autogra

Epoch,Training Loss,Validation Loss
1,0.201900,0.112250


✅ Phase phase3_mixed training complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm

def format_prompt(tweet):
    examples = (
        'Here are a few examples:\n'
        'Tweet: "Vaccines saved my life" Stance: "in-favor"\n'
        'Tweet: "Still deciding on those shots..." Stance: "neutral-or-unclear"\n'
        'Tweet: "This vaccine is poison." Stance: "Against"\n'
        'Now analyze:\n'
    )
    prompt = (
        f'What is the stance of the **author** of the following tweet toward COVID-19 vaccines?\n'
        f'Classify the stance as exactly one of: in-favor, against, neutral-or-unclear. '
        f'If the stance is unclear or mixed or if the tweet is off-topic, choose "neutral-or-unclear".\n'
        f'{examples}'
        f'Tweet: "{tweet}"\n'
        f'Respond with one exactly of the following options: "in-favor", "against", "neutral-or-unclear".'
    )
    return prompt

def predict_stances_batch(tweets, tokenizer, model):
    prompts = [format_prompt(tweet) for tweet in tweets]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10)
    predictions = [tokenizer.decode(output, skip_special_tokens=True).strip() for output in outputs]
    return predictions

In [ ]:
BATCH_SIZE = 32

# Batching predictions
from math import ceil

preds = []
labels = []
wrong = []

print("Generating predictions in batches...")

num_batches = ceil(len(df) / BATCH_SIZE)
for i in tqdm(range(num_batches)):
    batch_df = df.iloc[i * BATCH_SIZE: (i + 1) * BATCH_SIZE]
    batch_tweets = batch_df['tweet'].tolist()
    batch_labels = batch_df['label_majority'].tolist()

    batch_preds = predict_stances_batch(batch_tweets, tokenizer, model)

    preds.extend(batch_preds)
    labels.extend(batch_labels)

    for tweet, pred, label in zip(batch_tweets, batch_preds, batch_labels):
        if pred != label:
            wrong.append({
                "tweet": tweet,
                "label_majority": label,
                "prev_pred": pred
            })

df['predicted'] = preds

Generating predictions in batches...


  0%|          | 0/180 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
 97%|█████████▋| 174/180 [08:43<00:18,  3.12s/it]

In [ ]:
import pandas as pd

# Save entire DataFrame
df.to_csv("predictions.csv", index=False)

# Save incorrect predictions
wrong_df = pd.DataFrame(wrong)
wrong_df.to_csv("wrong_predictions.csv", index=False)

print("Saved predictions to predictions.csv")
print("Saved wrong predictions to wrong_predictions.csv")
